# 01 · Exploração do AIT-ADS

Objetivo: entender os alertas do **AIT Alert Data Set** (Wazuh e Suricata; o AMiner fica de
fora) para propor a **taxonomia interna de categorias** do Ronda e o mapeamento de fases de
ataque e assinaturas para essas categorias.

- Fonte dos alertas: `data/raw/ait-ads/json/<cenario>_wazuh.json` (Zenodo, registro 8263181).
- Rótulos: `data/raw/ait-ads/csv/<cenario>_alerts.txt` (repositório `ait-aecid/alert-data-set`,
  commit `5459b57`), gerados pelo `analyze.py` dos autores.
- Cenários: **russellmitchell** (análise) e **fox** (validação). A seção 7 usa só os CSVs dos oito
  cenários para checar a cobertura.

Como rodar: `uv sync --extra notebooks` e `uv run jupyter lab`, a partir da raiz. Os dados não
estão no git; veja `data/README.md`. O notebook é versionado **sem saídas**.

Os resultados e as decisões estão resumidos em `docs/exploracao-ait-ads.md`.

In [ ]:
import json
from collections import Counter
from itertools import islice
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 120)

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = RAIZ / "data" / "raw" / "ait-ads"
INTERIM = RAIZ / "data" / "interim" / "ait-ads"
INTERIM.mkdir(parents=True, exist_ok=True)

FASES_ATAQUE = [
    "network_scans",
    "service_scans",
    "wpscan",
    "dirb",
    "webshell",
    "cracking",
    "reverse_shell",
    "privilege_escalation",
    "service_stop",
    "dnsteal",
]
RUIDO = "false_positive"

## 1. Estrutura

### 1.1 CSV de alertas rotulados (`russellmitchell_alerts.txt`)

Colunas: `time` (Unix, segundos), `name` (descrição do alerta), `ip` (IP do agente), `host`
(máquina), `short` (código do tipo de alerta definido pelos autores; o prefixo indica a ferramenta:
A = AMiner, S = Suricata, W = Wazuh), `time_label` e `event_label`.

In [ ]:
def ler_csv(cenario: str) -> pd.DataFrame:
    csv = pd.read_csv(RAW / "csv" / f"{cenario}_alerts.txt")
    csv["ocorrido_em"] = pd.to_datetime(csv["time"], unit="s", utc=True)
    csv["ferramenta"] = csv["short"].str[0].map({"A": "aminer", "S": "suricata", "W": "wazuh"})
    return csv


csv_rm = ler_csv("russellmitchell")
print("alertas:", len(csv_rm))
print("período:", csv_rm["ocorrido_em"].min(), "→", csv_rm["ocorrido_em"].max())
print("\nfaltantes por coluna:")
print(csv_rm.isna().sum().to_string())
print("\npor ferramenta:")
print(csv_rm["ferramenta"].value_counts().to_string())

In [ ]:
pd.DataFrame(
    {
        "time_label": csv_rm["time_label"].value_counts(),
        "event_label": csv_rm["event_label"].value_counts(),
    }
).fillna(0).astype(int).sort_values("time_label", ascending=False)

Os dois rótulos têm significados diferentes, conforme o `analyze.py` dos autores:

- **`time_label`**: fase do ataque em cuja **janela de tempo** o alerta caiu, e
  `false_positive` fora de qualquer janela. Um alerta de rotina que por acaso ocorre durante um
  ataque recebe o rótulo da fase.
- **`event_label`**: rótulo **por evento**, derivado dos rótulos das linhas de log e dos fluxos de
  rede do AIT-LDSv2. Vale `-` quando o alerta não foi causado pelo ataque. Os nomes diferem um
  pouco das fases (`service_scan`, `crack_passwords`, `escalated_sudo_command` etc.).

Por isso, a classificação por janela (seção 4) vem acompanhada da contagem de eventos de ataque.

### 1.2 JSON do Wazuh (`russellmitchell_wazuh.json`)

A hipótese era um export do Elasticsearch com o alerta em `_source`. Lendo só as primeiras
linhas:

In [ ]:
CAMINHO_RM = RAW / "json" / "russellmitchell_wazuh.json"
with CAMINHO_RM.open(encoding="utf-8") as f:
    primeiras = [json.loads(linha) for linha in islice(f, 3)]

print("tipo de cada linha:", type(primeiras[0]).__name__)
print("chaves do 1º alerta:", sorted(primeiras[0]))
print("tem _source?", "_source" in primeiras[0])

**Formato real:** JSON Lines, **um alerta do Wazuh por linha, na raiz**, sem o envelope do
Elasticsearch. O Suricata chega pelo próprio Wazuh, que lê o `eve.json` (decoder `json`, regra
86601).

A seguir, um exemplo de cada ferramenta, procurado linha a linha:

In [ ]:
def primeiro(caminho: Path, condicao) -> dict:
    with caminho.open(encoding="utf-8") as f:
        for linha in f:
            alerta = json.loads(linha)
            if condicao(alerta):
                return alerta
    raise LookupError("nenhum alerta atende à condição")


def eh_suricata(a: dict) -> bool:
    return "suricata" in a["rule"].get("groups", []) and "alert" in a.get("data", {})


exemplo_wazuh = primeiro(CAMINHO_RM, lambda a: not eh_suricata(a))
exemplo_suricata = primeiro(CAMINHO_RM, eh_suricata)
print("=== Wazuh")
print(json.dumps(exemplo_wazuh, indent=2, ensure_ascii=False))

In [ ]:
print("=== Suricata (via Wazuh)")
print(json.dumps(exemplo_suricata, indent=2, ensure_ascii=False))

## 2. Achatar o JSON

Leitura linha a linha, extraindo só os campos úteis. Em seguida, os rótulos do CSV são
acoplados.

**Junção com o CSV.** O `analyze.py` escreve uma linha no CSV para cada alerta do JSON, na mesma
ordem, e o AMiner é escrito à parte. Então, filtrando o CSV sem o AMiner, a linha *i* do CSV é o
alerta *i* do JSON. A função abaixo **confere** isso, comparando horário, descrição e IP do agente
linha a linha, e falha se algo divergir.

In [ ]:
# Regras do Wazuh que reemitem, a partir do fast.log, alertas que o Suricata já gravou no eve.json.
# A 20161 só aparece nos outros cenários (confirmada no JSON do harrison).
REGRAS_FASTLOG = {"20100", "20101", "20151", "20152", "20161", "20162"}


def _ip(valor):
    # O fast.log grava "ip:porta" em dstip.
    if isinstance(valor, str) and valor.count(":") == 1:
        return valor.split(":")[0]
    return valor


def achatar(caminho: Path) -> pd.DataFrame:
    linhas = []
    with caminho.open(encoding="utf-8") as f:
        for linha in f:
            a = json.loads(linha)
            regra, dados = a["rule"], a.get("data", {})
            alerta = dados.get("alert", {})
            linhas.append(
                {
                    "ocorrido_em": a["@timestamp"],
                    "regra_id": regra["id"],
                    "regra_descricao": regra["description"],
                    "regra_nivel": regra["level"],
                    "regra_grupos": "|".join(regra.get("groups", [])),
                    "mitre_tecnicas": "|".join(regra.get("mitre", {}).get("id", [])),
                    "agente_nome": a["agent"]["name"],
                    "agente_ip": a["agent"]["ip"],
                    "decoder": a.get("decoder", {}).get("name"),
                    "local": a.get("location"),
                    "ip_origem": _ip(dados.get("src_ip") or dados.get("srcip")),
                    "ip_destino": _ip(dados.get("dest_ip") or dados.get("dstip")),
                    "porta_destino": dados.get("dest_port"),
                    "protocolo": dados.get("proto"),
                    "suricata_assinatura": alerta.get("signature"),
                    "suricata_assinatura_id": alerta.get("signature_id"),
                    "suricata_categoria": alerta.get("category"),
                    "suricata_severidade": alerta.get("severity"),
                }
            )
    df = pd.DataFrame(linhas)
    df["ocorrido_em"] = pd.to_datetime(df["ocorrido_em"], utc=True)
    df["regra_nivel"] = df["regra_nivel"].astype("int16")
    return df


EPOCA = pd.Timestamp(0, tz="UTC")


def rotular(df: pd.DataFrame, csv: pd.DataFrame) -> pd.DataFrame:
    ws = csv[csv["ferramenta"] != "aminer"].reset_index(drop=True)
    assert len(ws) == len(df), f"CSV tem {len(ws)} linhas W/S, JSON tem {len(df)}"
    nome = df["regra_descricao"].where(
        df["regra_descricao"].str.startswith("Suricata:"), "Wazuh: " + df["regra_descricao"]
    )
    iguais = (
        ((df["ocorrido_em"] - EPOCA) // pd.Timedelta(seconds=1) == ws["time"])
        & (nome == ws["name"])
        & (df["agente_ip"] == ws["ip"])
    )
    assert iguais.all(), f"{(~iguais).sum()} linhas não batem; a junção posicional não vale"
    df = df.copy()
    df[["short", "time_label", "event_label"]] = ws[["short", "time_label", "event_label"]]
    df["ferramenta"] = "wazuh"
    df.loc[df["suricata_assinatura"].notna(), "ferramenta"] = "suricata"
    df.loc[df["regra_id"].isin(REGRAS_FASTLOG), "ferramenta"] = "wazuh_fastlog"
    df["assinatura"] = df["suricata_assinatura"].where(
        df["ferramenta"] == "suricata", df["regra_id"] + " " + df["regra_descricao"]
    )
    df["grupos_ou_categoria"] = df["suricata_categoria"].where(
        df["ferramenta"] == "suricata", df["regra_grupos"]
    )
    return df

In [ ]:
rm = rotular(achatar(CAMINHO_RM), csv_rm)
rm.to_parquet(INTERIM / "russellmitchell_wazuh.parquet", index=False)
print(len(rm), "alertas →", INTERIM / "russellmitchell_wazuh.parquet")
rm.head(3).T

### 2.1 Campos sempre presentes e opcionais

Fração de alertas com o campo preenchido, por ferramenta. `wazuh_fastlog` são as cópias do
Suricata lidas do `fast.log` (seção 3.1).

In [ ]:
CAMPOS = [c for c in rm.columns if c not in ("short", "time_label", "event_label")]


def preenchido(s: pd.Series) -> pd.Series:
    return s.notna() & (s.astype(str) != "")


presenca = rm.groupby("ferramenta")[CAMPOS].apply(lambda d: d.apply(preenchido).mean()).T.round(3)
presenca["situacao"] = presenca.min(axis=1).map(lambda x: "sempre" if x == 1 else "opcional")
presenca.sort_values("situacao", ascending=False)

In [ ]:
# Varredura completa dos caminhos de chave do JSON, contando em quantos alertas cada um aparece.
def caminhos(obj: dict, prefixo: str = ""):
    for chave, valor in obj.items():
        caminho = f"{prefixo}.{chave}" if prefixo else chave
        if isinstance(valor, dict):
            yield from caminhos(valor, caminho)
        else:
            yield caminho


contagem = Counter()
with CAMINHO_RM.open(encoding="utf-8") as f:
    for linha in f:
        contagem.update(set(caminhos(json.loads(linha))))
chaves = pd.Series(contagem, name="alertas").sort_values(ascending=False)
print(f"{len(chaves)} caminhos; presentes em todos os {len(rm)} alertas:")
print(", ".join(chaves[chaves == len(rm)].index))
chaves.head(40).to_frame()

## 3. Volume e ruído

### 3.1 AMiner fora e o Suricata duplicado

O AMiner é descartado: é detector de anomalia sobre logs e está fora do escopo da primeira
versão. Entre Wazuh e Suricata há uma armadilha: **cada alerta do Suricata aparece duas vezes**,
uma pelo `eve.json` (regra 86601) e outra pelo `fast.log`, lida pelo decoder `snort` do Wazuh
(regras 201xx). O CSV dos autores conta a segunda cópia como alerta do Wazuh (`W-All-Evt`,
`W-All-Ids`, `W-All-Mul*`).

In [ ]:
print("descartados (AMiner):", (csv_rm["ferramenta"] == "aminer").sum())
print(rm["ferramenta"].value_counts().to_string())
fast = rm[rm["ferramenta"] == "wazuh_fastlog"]
print("\nregras do fast.log:")
print(fast.groupby(["regra_id", "regra_descricao"]).size().to_string())

In [ ]:
# Prova da duplicação: o fast.log grava "gid:sid:rev" em data.id; os sids batem com os do eve.json.
with CAMINHO_RM.open(encoding="utf-8") as f:
    sids_fast = Counter(
        a["data"]["id"].split(":")[1]
        for a in map(json.loads, f)
        if a["rule"]["id"] == "20101" and "id" in a.get("data", {})
    )
eve = rm[rm["ferramenta"] == "suricata"]
sids_eve = Counter(eve["suricata_assinatura_id"])
print(
    "alertas no eve.json:",
    sum(sids_eve.values()),
    "| regra 20101 (IDS event):",
    sum(sids_fast.values()),
)
print("todo sid do fast.log existe no eve.json:", set(sids_fast) <= set(sids_eve))
print(
    "contagem por sid igual ou menor no fast.log:",
    all(n <= sids_eve[s] for s, n in sids_fast.items()),
)

Daqui em diante, `wazuh_fastlog` fica fora das contagens por assinatura. **Consequência para o
Ronda:** o adaptador do Wazuh precisa descartar as regras 201xx do `fast.log` quando o Suricata
também chega pelo `eve.json`, ou o mesmo evento vira dois.

### 3.2 Contagem por ferramenta e top 20 assinaturas

In [ ]:
def sem_duplicatas(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["ferramenta"] != "wazuh_fastlog"]


print(sem_duplicatas(rm)["ferramenta"].value_counts().to_string())
(
    sem_duplicatas(rm)
    .groupby(["ferramenta", "assinatura"])
    .size()
    .rename("alertas")
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

### 3.3 Proporção de falsos positivos por ferramenta

Duas medidas: **fora das janelas de ataque** (`time_label = false_positive`, a definição dos
autores) e **não causado pelo ataque** (`event_label = -`).

In [ ]:
def falsos_positivos(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("ferramenta")
    return pd.DataFrame(
        {
            "alertas": g.size(),
            "fora_da_janela": g["time_label"].apply(lambda s: (s == RUIDO).mean()),
            "nao_causado_pelo_ataque": g["event_label"].apply(lambda s: (s == "-").mean()),
        }
    ).round(3)


falsos_positivos(rm)

## 4. Cruzamento com os ataques

### 4.1 Assinatura × fase

In [ ]:
def assinatura_x_fase(df: pd.DataFrame, top: int = 25) -> pd.DataFrame:
    d = sem_duplicatas(df)
    principais = d["assinatura"].value_counts().head(top).index
    tabela = pd.crosstab(d.loc[d["assinatura"].isin(principais), "assinatura"], d["time_label"])
    colunas = [c for c in [*FASES_ATAQUE, RUIDO] if c in tabela.columns]
    return tabela[colunas].loc[principais]


assinatura_x_fase(rm)

### 4.2 Classificação das assinaturas

Pela janela de tempo, como pedido:

- **só em ataques**: nenhuma ocorrência fora das janelas de ataque;
- **só ruído**: nenhuma ocorrência dentro de janelas de ataque;
- **ambos**: o resto.

A coluna `eventos_de_ataque` (`event_label ≠ -`) mostra quando um "ambos" é, na verdade, ruído
que só coincidiu com uma janela longa, como o Dovecot durante o `dnsteal`.

In [ ]:
CHAVE = ["ferramenta", "assinatura", "grupos_ou_categoria"]


def classificar_assinaturas(df: pd.DataFrame) -> pd.DataFrame:
    d = sem_duplicatas(df)
    g = d.groupby(CHAVE)
    t = pd.DataFrame(
        {
            "total": g.size(),
            "ruido": g["time_label"].apply(lambda s: (s == RUIDO).sum()),
            "eventos_de_ataque": g["event_label"].apply(lambda s: (s != "-").sum()),
        }
    )
    t["em_ataque"] = t["total"] - t["ruido"]
    em_ataque = d[d["time_label"] != RUIDO].groupby(CHAVE)["time_label"]
    t["fase_dominante"] = em_ataque.agg(lambda s: s.value_counts().idxmax())
    t["fases"] = em_ataque.agg(lambda s: ",".join(sorted(s.unique())))
    t["classificacao"] = "ambos"
    t.loc[t["ruido"] == 0, "classificacao"] = "só em ataques"
    t.loc[t["em_ataque"] == 0, "classificacao"] = "só ruído"
    t["fase_dominante"] = t["fase_dominante"].fillna("-")
    t["fases"] = t["fases"].fillna("-")
    colunas = ["total", "em_ataque", "ruido", "eventos_de_ataque", "fase_dominante", "fases"]
    return t[[*colunas, "classificacao"]].sort_values("total", ascending=False).reset_index()


class_rm = classificar_assinaturas(rm)
print(class_rm["classificacao"].value_counts().to_string())
class_rm

## 5. Taxonomia

### 5.1 Categorias internas

| Categoria | Significado |
| --- | --- |
| `reconhecimento` | Varreduras de rede, de serviço e de aplicação web |
| `ataque_web` | Exploração de aplicação web (injeção, webshell) |
| `forca_bruta` | Tentativas repetidas de autenticação e quebra de senhas |
| `comprometimento` | Execução no host: shell reverso, escalada de privilégio |
| `negacao_servico` | Indisponibilidade provocada: parada de serviço, DoS |
| `exfiltracao` | Saída de dados, como o túnel DNS do `dnsteal` |
| `benigno` | Atividade legítima **pela semântica** do alerta (atualização, login bem-sucedido) |
| `desconhecido` | Sem regra de mapeamento, ou alerta sem semântica de ataque (anomalia de protocolo) |

`benigno` descreve **o que o alerta diz**, não quanto ele é barulhento. Uma assinatura que só
apareceu como ruído mantém a categoria da sua semântica e entra na lista de **candidatas a ajuste
de política** (5.4).

### 5.2 Fase → categoria

In [ ]:
FASE_PARA_CATEGORIA = {
    "network_scans": "reconhecimento",
    "service_scans": "reconhecimento",
    "wpscan": "reconhecimento",
    "dirb": "reconhecimento",
    "webshell": "ataque_web",
    "cracking": "forca_bruta",
    "reverse_shell": "comprometimento",
    "privilege_escalation": "comprometimento",
    "service_stop": "negacao_servico",
    "dnsteal": "exfiltracao",
}
pd.Series(FASE_PARA_CATEGORIA, name="categoria").to_frame()

### 5.3 Assinatura → categoria

As regras seguem a ordem de preferência pedida: **grupos de regra do Wazuh** e **categoria do
Suricata** primeiro, depois a **família ET** da assinatura (o prefixo `ET SCAN`, `ET POLICY`...,
que é convenção estruturada do conjunto Emerging Threats). Por último vêm as **exceções por regra
ou assinatura**, cada uma com o motivo e a evidência nos dados.

In [ ]:
# Wazuh: o primeiro grupo que casar decide (ordem importa).
GRUPOS_WAZUH = [
    ("sudo", "comprometimento"),
    ("su", "comprometimento"),
    ("authentication_failed", "forca_bruta"),
    ("authentication_failures", "forca_bruta"),
    ("invalid_login", "forca_bruta"),
    ("web_scan", "reconhecimento"),
    ("recon", "reconhecimento"),
    ("access_denied", "reconhecimento"),
    ("attack", "ataque_web"),
    ("authentication_success", "benigno"),
]
# Suricata: família ET da assinatura, antes da categoria.
FAMILIA_ET = {"SCAN": "reconhecimento", "POLICY": "benigno"}
CATEGORIA_SURICATA = {
    "Not Suspicious Traffic": "benigno",
    "Web Application Attack": "ataque_web",
    "Generic Protocol Command Decode": "desconhecido",
    "Potentially Bad Traffic": "desconhecido",
}
# Exceções: (categoria, motivo). Chave = id da regra do Wazuh ou assinatura do Suricata.
EXCECOES = {
    "31101": (
        "reconhecimento",
        "grupo 'attack', mas um erro 400 isolado é rastro de varredura: >99% em dirb/wpscan",
    ),
    "31509": (
        "forca_bruta",
        "grupos 'appsec|attack', mas é tentativa de login no CMS (vizinha da 31510, brute force)",
    ),
    "52507": ("benigno", "grupo 'virus', mas é a atualização de rotina do ClamAV (nível 3)"),
    "5502": ("benigno", "fim de sessão PAM; só indica ataque combinado com outros eventos"),
    "31122": ("desconhecido", "erro 500 do servidor: pode ser sintoma de exploração ou falha"),
    "80730": ("desconhecido", "checagem de permissão do SELinux, sem semântica de ataque"),
    "ET INFO Observed DNS Query to .biz TLD": (
        "exfiltracao",
        "é o alerta que dispara com o dnsteal em harrison e santos (seção 7)",
    ),
}


def categorizar(ferramenta: str, assinatura: str, grupos: str) -> tuple[str, str]:
    if ferramenta == "wazuh_fastlog":
        return "-", "cópia do alerta do Suricata; descartar na ingestão"
    chave = assinatura.split(" ")[0] if ferramenta == "wazuh" else assinatura
    if chave in EXCECOES:
        return EXCECOES[chave]
    if ferramenta == "wazuh":
        conjunto = set(grupos.split("|"))
        for grupo, categoria in GRUPOS_WAZUH:
            if grupo in conjunto:
                return categoria, f"grupo {grupo}"
        return "desconhecido", "nenhum grupo mapeado"
    partes = assinatura.split(" ")
    if partes[0] == "ET" and len(partes) > 1 and partes[1] in FAMILIA_ET:
        return FAMILIA_ET[partes[1]], f"família ET {partes[1]}"
    if grupos in CATEGORIA_SURICATA:
        return CATEGORIA_SURICATA[grupos], f"categoria {grupos}"
    return "desconhecido", "categoria do Suricata sem mapeamento"


def aplicar_taxonomia(tabela: pd.DataFrame) -> pd.DataFrame:
    t = tabela.copy()
    resultado = [
        categorizar(f, a, g)
        for f, a, g in t[["ferramenta", "assinatura", "grupos_ou_categoria"]].itertuples(
            index=False
        )
    ]
    t["categoria_proposta"] = [c for c, _ in resultado]
    t["motivo"] = [m for _, m in resultado]
    t["categoria_da_fase"] = t["fase_dominante"].map(FASE_PARA_CATEGORIA).fillna("-")
    return t


tax_rm = aplicar_taxonomia(class_rm)
tax_rm[
    ["ferramenta", "assinatura", "classificacao", "fase_dominante", "categoria_proposta", "motivo"]
]

**Validação contra os dados.** Para as assinaturas que aparecem em ataques e têm eventos de
ataque de fato, a categoria proposta deveria coincidir com a categoria da fase dominante. As
divergências abaixo são decisões conscientes, justificadas no documento, ou problemas a
revisar.

In [ ]:
def divergencias(tax: pd.DataFrame) -> pd.DataFrame:
    com_ataque = tax[(tax["classificacao"] != "só ruído") & (tax["eventos_de_ataque"] > 0)]
    return com_ataque[com_ataque["categoria_proposta"] != com_ataque["categoria_da_fase"]][
        [
            "ferramenta",
            "assinatura",
            "eventos_de_ataque",
            "fase_dominante",
            "categoria_da_fase",
            "categoria_proposta",
            "motivo",
        ]
    ]


divergencias(tax_rm)

### 5.4 Candidatas a ajuste de política

Assinaturas "só ruído", mais as "ambos" sem nenhum evento de ataque. **Não viram `benigno`**:
ficam com a categoria da semântica, e a política do tenant decide se rebaixa a urgência. A seção
7 mostra por que isso importa: um alerta que é só ruído aqui é o detector de exfiltração em outro
cenário.

In [ ]:
def candidatas(tax: pd.DataFrame) -> pd.DataFrame:
    filtro = (tax["classificacao"] == "só ruído") | (
        (tax["classificacao"] == "ambos") & (tax["eventos_de_ataque"] == 0)
    )
    return tax[filtro][["ferramenta", "assinatura", "total", "classificacao", "categoria_proposta"]]


candidatas(tax_rm)

## 6. Validação no cenário `fox`

Mesmos passos (3 a 5) no fox, que tem 462 mil alertas Wazuh/Suricata, quase todos do dirb.

In [ ]:
csv_fox = ler_csv("fox")
fox = rotular(achatar(RAW / "json" / "fox_wazuh.json"), csv_fox)
fox.to_parquet(INTERIM / "fox_wazuh.parquet", index=False)
print("alertas W/S:", len(fox), "| AMiner descartados:", (csv_fox["ferramenta"] == "aminer").sum())
print(fox["ferramenta"].value_counts().to_string())

In [ ]:
pd.concat({"russellmitchell": falsos_positivos(rm), "fox": falsos_positivos(fox)}, axis=1)

In [ ]:
class_fox = classificar_assinaturas(fox)
tax_fox = aplicar_taxonomia(class_fox)
print(class_fox["classificacao"].value_counts().to_string())
assinatura_x_fase(fox)

In [ ]:
comparacao = tax_rm.merge(
    tax_fox, on=["ferramenta", "assinatura"], how="outer", suffixes=("_rm", "_fox"), indicator=True
)
print("só no russellmitchell:", (comparacao["_merge"] == "left_only").sum())
print("só no fox:", (comparacao["_merge"] == "right_only").sum())
print("em ambos:", (comparacao["_merge"] == "both").sum())
mudou = comparacao[
    (comparacao["_merge"] == "both")
    & (comparacao["classificacao_rm"] != comparacao["classificacao_fox"])
]
mudou[
    [
        "ferramenta",
        "assinatura",
        "total_rm",
        "classificacao_rm",
        "total_fox",
        "classificacao_fox",
        "eventos_de_ataque_fox",
    ]
]

In [ ]:
print("assinaturas novas no fox:")
comparacao[comparacao["_merge"] == "right_only"][
    ["ferramenta", "assinatura", "total_fox", "classificacao_fox", "categoria_proposta_fox"]
]

In [ ]:
divergencias(tax_fox)

In [ ]:
candidatas(tax_fox)

## 7. Cobertura nos oito cenários (só CSV)

Checagem barata com os CSVs dos oito cenários: quais fases os detectores enxergam e quais
assinaturas ficaram fora da análise dos dois cenários.

In [ ]:
todos = pd.concat(
    [
        ler_csv(p.name.split("_")[0]).assign(cenario=p.name.split("_")[0])
        for p in sorted((RAW / "csv").glob("*_alerts.txt"))
    ],
    ignore_index=True,
)
print("eventos de ataque (event_label) por ferramenta, oito cenários:")
eventos = todos[todos["event_label"] != "-"]
pd.crosstab(eventos["event_label"], eventos["ferramenta"])

In [ ]:
biz = todos[todos["name"] == "Suricata: Alert - ET INFO Observed DNS Query to .biz TLD"]
pd.crosstab(biz["cenario"], biz["event_label"])

In [ ]:
vistos = set(todos.loc[todos["cenario"].isin(["russellmitchell", "fox"]), "name"])
outros = todos[(todos["ferramenta"] != "aminer") & ~todos["name"].isin(vistos)]
outros.groupby("name").agg(
    alertas=("time", "size"),
    cenarios=("cenario", lambda s: ",".join(sorted(s.unique()))),
    em_ataque=("time_label", lambda s: (s != RUIDO).sum()),
    eventos_de_ataque=("event_label", lambda s: (s != "-").sum()),
).sort_values("alertas", ascending=False)

## 8. Rascunho da taxonomia

Tabela única com russellmitchell e fox juntos, gravada em `docs/taxonomia-rascunho.csv`. As
cópias do `fast.log` entram com classificação `duplicata`, para o adaptador saber que deve
descartá-las.

In [ ]:
ambos_cenarios = pd.concat([rm, fox], ignore_index=True)
tax = aplicar_taxonomia(classificar_assinaturas(ambos_cenarios))

fast_log = (
    ambos_cenarios[ambos_cenarios["ferramenta"] == "wazuh_fastlog"]
    .groupby(CHAVE)
    .size()
    .rename("total")
    .reset_index()
    .assign(fase_dominante="-", classificacao="duplicata", categoria_proposta="-")
)
rascunho = pd.concat([tax, fast_log], ignore_index=True)[
    [
        "ferramenta",
        "assinatura",
        "grupos_ou_categoria",
        "fase_dominante",
        "classificacao",
        "categoria_proposta",
    ]
]
rascunho = rascunho.sort_values(["ferramenta", "categoria_proposta", "assinatura"])
rascunho.to_csv(RAIZ / "docs" / "taxonomia-rascunho.csv", index=False)
print(len(rascunho), "assinaturas →", RAIZ / "docs" / "taxonomia-rascunho.csv")
rascunho.groupby(["categoria_proposta", "classificacao"]).size().unstack(fill_value=0)